# XGBoost & TensorFlow MLP — Hospital Readmission Prediction
**Thesis:** Smart Healthcare Systems — ML for Hospital Readmission Prediction  
**Dataset:** Diabetes 130-US Hospitals (Kaggle)  
**Models:** XGBoost (Optuna) + MLP Neural Network (TensorFlow/Keras)  

---

### Pipeline features:
1. **Regularization parameters** (gamma, reg_alpha, reg_lambda) in Optuna search space
2. **Overfitting monitoring** — Train vs Validation vs Test metrics
3. **SHAP interpretability** — Explaining model decisions
4. **Proper Calibration** — Cross-validated instead of prefit on validation set
5. **Bootstrap Confidence Intervals** — 95% CI for all metrics
6. **Interaction features** — Clinically relevant new features
7. **TensorFlow MLP** — 3-layer neural network comparison


## 1. Data Loading & Library Installation

In [ ]:
!pip install optuna shap -q

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, precision_recall_curve, roc_curve,
    confusion_matrix
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

print('Loading data...')
df = pd.read_csv('diabetic_data.csv')

# Replace '?' with NaN for native XGBoost handling
df.replace('?', np.nan, inplace=True)
print(f'Dataset loaded: {df.shape}')
print(f'Readmission classes: {df["readmitted"].value_counts().to_dict()}')


## 2. Feature Engineering

Creating clinically relevant features:
- ICD-9 diagnosis grouping into clinical categories
- Composite features (total visits, medication changes, procedures/day)
- **NEW:** Interaction features between clinically relevant variables
- **NEW:** Binary flags for missing data (medical_specialty, weight)


In [ ]:
print('Applying Feature Engineering...')
df_fe = df.copy()

#  Create Target 
df_fe['target'] = (df_fe['readmitted'] == '<30').astype(int)
print(f'Target distribution: {df_fe["target"].value_counts().to_dict()}')
print(f'Readmission <30 days rate: {df_fe["target"].mean():.3%}')

#  1. ICD-9 Diagnosis Grouping 
def map_diag(code):
    try:
        code = str(code).strip()
        if code.startswith('V') or code.startswith('E'):
            return 'external'
        c = float(code)
        if 250 <= c < 251:   return 'diabetes'
        if 390 <= c <= 459:  return 'circulatory'
        if 460 <= c <= 519:  return 'respiratory'
        if 520 <= c <= 579:  return 'digestive'
        if 580 <= c <= 629:  return 'genitourinary'
        if 710 <= c <= 739:  return 'musculoskeletal'
        if 800 <= c <= 999:  return 'injury'
        if 140 <= c <= 239:  return 'neoplasms'
        return 'other'
    except:
        return 'other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df_fe[f'{col}_cat'] = df_fe[col].apply(map_diag)

#  2. Base composite features 
df_fe['total_prior_visits'] = (
    df_fe['number_outpatient'] + df_fe['number_emergency'] + df_fe['number_inpatient']
)

med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'glipizide', 'glyburide', 'pioglitazone',
            'rosiglitazone', 'acarbose', 'insulin']

df_fe['num_med_changes'] = df_fe[med_cols].apply(
    lambda row: sum(1 for v in row if v in ['Up', 'Down']), axis=1
)

df_fe['procedures_per_day'] = df_fe['num_procedures'] / (df_fe['time_in_hospital'] + 1)

age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35, '[40-50)': 45,
    '[50-60)': 55, '[60-70)': 65, '[70-80)': 75, '[80-90)': 85, '[90-100)': 95
}
df_fe['age_numeric'] = df_fe['age'].map(age_map)

#  3. [NEW] Interaction Features 
df_fe['age_x_medications'] = df_fe['age_numeric'] * df_fe['num_medications']
df_fe['hospital_x_diagnoses'] = df_fe['time_in_hospital'] * df_fe['number_diagnoses']
df_fe['age_x_inpatient'] = df_fe['age_numeric'] * df_fe['number_inpatient']
df_fe['meds_per_diagnosis'] = df_fe['num_medications'] / (df_fe['number_diagnoses'] + 1)

#  4. [NEW] Binary flags for missing data 
df_fe['has_weight'] = df_fe['weight'].notna().astype(int)
df_fe['has_specialty'] = df_fe['medical_specialty'].notna().astype(int)
df_fe['has_payer'] = df_fe['payer_code'].notna().astype(int)

#  5. Drop columns not needed 
drop_cols = [
    'encounter_id', 'readmitted', 'weight', 'payer_code',
    'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'age'
]
df_fe.drop(columns=drop_cols, inplace=True)

print(f'\nFeature engineering complete!')
print(f'Final columns: {df_fe.shape[1]}')
print(f'New features: age_x_medications, hospital_x_diagnoses, age_x_inpatient, '
      f'meds_per_diagnosis, has_weight, has_specialty, has_payer')


## 3. Categorical Variable Encoding

Using XGBoost's native categorical support (no one-hot encoding) — reduces feature count and avoids curse of dimensionality.


In [ ]:
print('Converting object/string columns to category dtype. ')

categorical_cols = df_fe.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    df_fe[col] = df_fe[col].astype('category')

print(f'Converted {len(categorical_cols)} columns to categorical.')
print(f'Categorical: {categorical_cols[:10]}{"..." if len(categorical_cols) > 10 else ""}')


## 4. Data Splitting — Patient-Level

**Critical:** Using `GroupShuffleSplit` with `patient_nbr` so encounters from the same patient are never split across sets (prevents data leakage).  
Split: **64% Train / 16% Validation / 20% Test**


In [ ]:
print('Creating Train / Validation / Test splits at patient level. \n')

X = df_fe.drop(columns=['target'])
y = df_fe['target']
groups = df_fe['patient_nbr']

# Step 1: 80% Train+Val / 20% Test
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_val_idx, test_idx = next(gss1.split(X, y, groups))

X_temp = X.iloc[train_val_idx]
y_temp = y.iloc[train_val_idx]
groups_temp = groups.iloc[train_val_idx]
X_test = X.iloc[test_idx].drop(columns=['patient_nbr'])
y_test = y.iloc[test_idx]

# Step 2: 80% Train / 20% Val (from temp)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, val_idx = next(gss2.split(X_temp, y_temp, groups_temp))

X_train = X_temp.iloc[train_idx].drop(columns=['patient_nbr'])
y_train = y_temp.iloc[train_idx]

X_val = X_temp.iloc[val_idx].drop(columns=['patient_nbr'])
y_val = y_temp.iloc[val_idx]

print(f'Train:      {X_train.shape[0]:>6} samples ({y_train.mean():.2%} positive)')
print(f'Validation: {X_val.shape[0]:>6} samples ({y_val.mean():.2%} positive)')
print(f'Test:       {X_test.shape[0]:>6} samples ({y_test.mean():.2%} positive)')
print(f'\nFeatures:   {X_train.shape[1]}')
print(f'\n  No patient appears in more than 1 set (GroupShuffleSplit)')


## 5. Hyperparameter Tuning — Optuna

**Improvements:**
- Extended search space: now includes `gamma`, `reg_alpha`, `reg_lambda` for regularization
- 50 trials (instead of 30) for better exploration
- Objective: Maximize AUPRC (appropriate for imbalanced data)


In [ ]:
print('Starting Hyperparameter Tuning with Optuna (50 trials)...')
print('This will take approximately 5-10 minutes.\n')

neg_cases = (y_train == 0).sum()
pos_cases = (y_train == 1).sum()
scale_pos = neg_cases / pos_cases
print(f'Class ratio: {neg_cases}:{pos_cases} (scale_pos_weight = {scale_pos:.2f})\n')

def objective(trial):
    param = {
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        # [NEW] Regularization parameters
        'gamma': trial.suggest_float('gamma', 0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        # Fixed parameters
        'scale_pos_weight': scale_pos,
        'enable_categorical': True,
        'tree_method': 'hist',
        'random_state': 42,
        'eval_metric': 'aucpr',
        'early_stopping_rounds': 30
    }

    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    preds = model.predict_proba(X_val)[:, 1]
    aucpr = average_precision_score(y_val, preds)
    return aucpr

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print('\n' + '=' * 55)
print('  OPTUNA TUNING COMPLETE')
print('=' * 55)
print(f'  Best AUPRC (Validation): {study.best_value:.4f}')
print(f'  Number of trials: {len(study.trials)}')
print('\n  Best Parameters:')
for k, v in study.best_params.items():
    print(f'    {k:>20s}: {v}')
print('=' * 55)


## 6. Training Final Model

Training with best parameters and early stopping.

In [ ]:
print('Training the final model with best parameters. \n')

best_params = study.best_params.copy()
best_params['n_estimators'] = 1000
best_params['scale_pos_weight'] = scale_pos
best_params['enable_categorical'] = True
best_params['tree_method'] = 'hist'
best_params['random_state'] = 42
best_params['eval_metric'] = 'aucpr'
best_params['early_stopping_rounds'] = 50

final_xgb = xgb.XGBClassifier(**best_params)
final_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

print(f'Final model: {final_xgb.best_iteration} trees (early stopped from 1000)')
print(f'Best validation AUPRC: {final_xgb.best_score:.4f}')


## 6b. Save Model Files for ML Service
Saves the model and feature metadata needed by the FastAPI microservice.

In [ ]:
import joblib
import os

os.makedirs("ml_models", exist_ok=True)

# Save the uncalibrated XGBoost model
# (CalibratedClassifierCV is not supported by shap.TreeExplainer)
joblib.dump(final_xgb, "ml_models/xgboost_model.pkl")
print(" xgboost_model.pkl saved")

# Save feature names (column order matters for inference)
feature_names = X_train.columns.tolist()
joblib.dump(feature_names, "ml_models/feature_names.pkl")
print(f" feature_names.pkl saved — {len(feature_names)} features")
print(feature_names[:10], "...")

# Save categorical column names so the ML service knows which to cast
cat_cols = [c for c in feature_names if X_train[c].dtype.name == "category"]
joblib.dump(cat_cols, "ml_models/categorical_cols.pkl")
print(f" categorical_cols.pkl saved — {len(cat_cols)} categorical features")


## 7. Calibration — Cross-Validated

**Improvement:** Using `cv=3` instead of `cv='prefit'` to avoid reusing the validation set already used for early stopping.


In [ ]:
print('Running Cross-Validated Calibration...\n')

# New uncalibrated model with same parameters
# (without early_stopping_rounds since CV calibration does internal fit)
cal_params = best_params.copy()
cal_params.pop('early_stopping_rounds', None)
cal_params['n_estimators'] = final_xgb.best_iteration  # Use tree count from early stopping

base_model_for_cal = xgb.XGBClassifier(**cal_params)

final_calibrated = CalibratedClassifierCV(
    estimator=base_model_for_cal,
    method='isotonic',
    cv=3  # 3-fold CV instead of prefit avoids validation set reuse
)
final_calibrated.fit(X_train, y_train)

print(' Calibration complete (3-fold CV isotonic)')
print('  Calibration did not use the validation set for avoiding optimistic bias.')


## 8. Test Set Evaluation

### 8a. Core Metrics + Overfitting Monitoring

In [ ]:
print('Computing metrics on Train / Validation / Test. \n')

# Predictions on each set
y_train_proba = final_calibrated.predict_proba(X_train)[:, 1]
y_val_proba = final_calibrated.predict_proba(X_val)[:, 1]
y_test_proba = final_calibrated.predict_proba(X_test)[:, 1]

# Metrics for each set
sets = {
    'Train': (y_train, y_train_proba),
    'Validation': (y_val, y_val_proba),
    'Test': (y_test, y_test_proba)
}

results = {}
print('=' * 65)
print(f'  {"Set":<12} {"AUROC":>8} {"AUPRC":>8} {"Brier":>8}')
print('=' * 65)

for name, (y_true, y_prob) in sets.items():
    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)
    results[name] = {'AUROC': auroc, 'AUPRC': auprc, 'Brier': brier}
    print(f'  {name:<12} {auroc:>8.4f} {auprc:>8.4f} {brier:>8.4f}')

print('=' * 65)

gap_auroc = results['Train']['AUROC'] - results['Test']['AUROC']
gap_auprc = results['Train']['AUPRC'] - results['Test']['AUPRC']

print(f'\n  Overfitting Gap (Train - Test):')
print(f'    AUROC: {gap_auroc:+.4f}', end='')
print(' ✓ Minimal overfitting' if gap_auroc < 0.05 else ('  Moderate overfitting' if gap_auroc < 0.10 else '  Significant overfitting'))
print(f'    AUPRC: {gap_auprc:+.4f}', end='')
print(' ✓ Minimal overfitting' if gap_auprc < 0.05 else ('  Moderate overfitting' if gap_auprc < 0.10 else '  Significant overfitting'))


### 8b. Bootstrap Confidence Intervals (95%)

In [ ]:
print('Computing 95% Confidence Intervals via Bootstrap (1000 iterations). \n')

n_bootstrap = 1000
np.random.seed(42)
boot_auroc, boot_auprc, boot_brier = [], [], []

for i in range(n_bootstrap):
    idx = resample(range(len(y_test)), random_state=i)
    y_true_b = y_test.iloc[idx]
    y_prob_b = y_test_proba[idx]
    if len(y_true_b.unique()) < 2: continue
    boot_auroc.append(roc_auc_score(y_true_b, y_prob_b))
    boot_auprc.append(average_precision_score(y_true_b, y_prob_b))
    boot_brier.append(brier_score_loss(y_true_b, y_prob_b))

auroc_ci = np.percentile(boot_auroc, [2.5, 97.5])
auprc_ci = np.percentile(boot_auprc, [2.5, 97.5])
brier_ci = np.percentile(boot_brier, [2.5, 97.5])

test_auroc = results['Test']['AUROC']
test_auprc = results['Test']['AUPRC']
test_brier = results['Test']['Brier']

print('=' * 65)
print('  FINAL TEST RESULTS WITH 95% CONFIDENCE INTERVALS')
print('=' * 65)
print(f'  AUROC:  {test_auroc:.4f}  (95% CI: {auroc_ci[0]:.4f} – {auroc_ci[1]:.4f})')
print(f'  AUPRC:  {test_auprc:.4f}  (95% CI: {auprc_ci[0]:.4f} – {auprc_ci[1]:.4f})')
print(f'  Brier:  {test_brier:.4f}  (95% CI: {brier_ci[0]:.4f} – {brier_ci[1]:.4f})')
print('=' * 65)
print(f'\n  Bootstrap samples: {len(boot_auroc)} / {n_bootstrap}')


### 8c. Optimal Threshold & Classification Report

In [ ]:
# Dynamic threshold based on F1-Score
precisions, recalls, thresholds = precision_recall_curve(y_test, y_test_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f'Optimal threshold based on F1-Score: {optimal_threshold:.4f}')
print(f'F1 at this threshold: {f1_scores[optimal_idx]:.4f}\n')

y_pred_optimal = (y_test_proba >= optimal_threshold).astype(int)
print(f'Classification Report (threshold = {optimal_threshold:.4f}):')
print(classification_report(y_test, y_pred_optimal, target_names=['No Readmit', 'Readmit <30d']))

cm = confusion_matrix(y_test, y_pred_optimal)
print('Confusion Matrix:')
print(cm)
print(f'\nInterpretation:')
print(f'  True Negatives:  {cm[0,0]:>5} (correct NO readmission)')
print(f'  False Positives: {cm[0,1]:>5} (false alerts)')
print(f'  False Negatives: {cm[1,0]:>5} (missed readmissions)')
print(f'  True Positives:  {cm[1,1]:>5} (correct readmission alerts)')


## 9. Visualizations

### 9a. ROC Curve & Precision-Recall Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

#  ROC Curve 
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
axes[0].plot(fpr, tpr, color='#2563eb', lw=2,
             label=f'XGBoost (AUROC = {test_auroc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUROC = 0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#2563eb')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

#  Precision-Recall Curve 
axes[1].plot(recalls, precisions, color='#dc2626', lw=2,
             label=f'XGBoost (AUPRC = {test_auprc:.4f})')
baseline_pr = y_test.mean()
axes[1].axhline(y=baseline_pr, color='k', linestyle='--', lw=1, alpha=0.5,
                label=f'Baseline ({baseline_pr:.3f})')
axes[1].fill_between(recalls, precisions, alpha=0.1, color='#dc2626')
# Mark optimal threshold
axes[1].scatter([recalls[optimal_idx]], [precisions[optimal_idx]],
                color='#dc2626', s=100, zorder=5,
                label=f'Optimal F1 (t={optimal_threshold:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Saved: roc_pr_curves.png")

### 9b. Calibration Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# Calibration curve of the calibrated model
prob_true_cal, prob_pred_cal = calibration_curve(y_test, y_test_proba, n_bins=10, strategy='uniform')

# Calibration curve of the uncalibrated model for comparison
y_test_proba_uncal = final_xgb.predict_proba(X_test)[:, 1]
prob_true_uncal, prob_pred_uncal = calibration_curve(y_test, y_test_proba_uncal, n_bins=10, strategy='uniform')

ax.plot(prob_pred_cal, prob_true_cal, 's-', color='#2563eb', lw=2,
        label=f'Calibrated (Brier={test_brier:.4f})')
ax.plot(prob_pred_uncal, prob_true_uncal, 'o--', color='#9ca3af', lw=1.5,
        label=f'Uncalibrated (Brier={brier_score_loss(y_test, y_test_proba_uncal):.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Observed Frequency')
ax.set_title('Calibration Curve')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Saved: calibration_curve.png")

### 9c. Confusion Matrix Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Readmit', 'Readmit <30d'],
            yticklabels=['No Readmit', 'Readmit <30d'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix (Threshold = {optimal_threshold:.4f})')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Saved: confusion_matrix.png")

## 10. SHAP — Model Interpretability

**Why SHAP instead of feature_importances_:**
- The default `feature_importances_` of XGBoost only measures how often a feature is used (gain-based)
- SHAP provides **directional impact**: which feature increases or decreases readmission risk
- Critical for clinical interpretability — clinicians need to know **why**


In [ ]:
print('Computing SHAP values (may take 1-2 minutes). \n')

# Using the uncalibrated model — SHAP does not support CalibratedClassifierCV
explainer = shap.TreeExplainer(final_xgb)

# Compute on subset of test set (faster)
X_test_sample = X_test.sample(n=min(2000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_test_sample)

print(f'SHAP values computed for {len(X_test_sample)} test samples.')


In [ ]:
#  SHAP Summary Plot (Beeswarm) 
print('Top 20 features by importance (SHAP):')
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_sample, max_display=20, show=False)
plt.title('SHAP Feature Importance — Impact on Readmission Prediction')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Saved: shap_summary.png')


In [ ]:
#  SHAP Bar Plot (Mean Absolute) 
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_sample, plot_type='bar', max_display=20, show=False)
plt.title('Mean |SHAP| — Feature Importance Ranking')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Saved: shap_bar.png')


In [ ]:
#  SHAP Dependence Plots for top 3 features 
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top3_idx = np.argsort(mean_abs_shap)[-3:][::-1]
top3_features = X_test_sample.columns[top3_idx].tolist()

# FIX: Convert categorical to codes  SHAP dependence_plot
# uses numpy sorting internally which fails on mixed types
X_test_display = X_test_sample.copy()
for col in X_test_display.select_dtypes(['category']).columns:
    X_test_display[col] = X_test_display[col].cat.codes

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat in enumerate(top3_features):
    shap.dependence_plot(feat, shap_values, X_test_display, ax=axes[i], show=False, interaction_index=None)
    axes[i].set_title(f'SHAP Dependence: {feat}')

plt.tight_layout()
plt.savefig('shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print(f' Saved: shap_dependence.png')
print(f'  Top 3 features: {top3_features}')


## 11. Optuna Visualization — Hyperparameter Analysis

In [ ]:
# Optuna trials history
trials_df = study.trials_dataframe()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimization history
axes[0].plot(trials_df.index, trials_df['value'], 'o-', alpha=0.5, markersize=4, color='#6366f1')
axes[0].axhline(y=study.best_value, color='#dc2626', linestyle='--', lw=1,
                label=f'Best: {study.best_value:.4f}')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('AUPRC (Validation)')
axes[0].set_title('Optuna Optimization History')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Parameter importance (based on correlation with performance)
param_cols = [c for c in trials_df.columns if c.startswith('params_')]
correlations = {}
for col in param_cols:
    try:
        corr = trials_df[col].astype(float).corr(trials_df['value'])
        correlations[col.replace('params_', '')] = abs(corr)
    except:
        pass

corr_df = pd.DataFrame.from_dict(correlations, orient='index', columns=['Correlation'])
corr_df = corr_df.sort_values('Correlation', ascending=True)
axes[1].barh(corr_df.index, corr_df['Correlation'], color='#6366f1', alpha=0.7)
axes[1].set_xlabel('|Correlation| with AUPRC')
axes[1].set_title('Hyperparameter Sensitivity')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('optuna_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Saved: optuna_analysis.png")

## 12. TensorFlow Neural Network — Comparison with XGBoost
> **MLP (Multi-Layer Perceptron)** with TensorFlow/Keras  
> Since XGBoost uses native categorical features, the MLP needs one-hot encoding + scaling  
> Goal: Compare deep learning approach vs tree-based ML on the same tabular dataset

In [ ]:
#  TensorFlow Imports & Data Preparation 
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import joblib

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

tf.random.set_seed(42)
np.random.seed(42)

#  One-Hot Encoding for TF (XGBoost used native categoricals) ──
print('\nPreparing one-hot encoded data for TensorFlow...')
X_train_ohe = pd.get_dummies(X_train, drop_first=True)
X_val_ohe   = pd.get_dummies(X_val, drop_first=True)
X_test_ohe  = pd.get_dummies(X_test, drop_first=True)

X_val_ohe  = X_val_ohe.reindex(columns=X_train_ohe.columns, fill_value=0)
X_test_ohe = X_test_ohe.reindex(columns=X_train_ohe.columns, fill_value=0)

for df_ in [X_train_ohe, X_val_ohe, X_test_ohe]:
    df_.columns = df_.columns.astype(str).str.replace('[^A-Za-z0-9_]+', '_', regex=True)

train_median = X_train_ohe.median(numeric_only=True)
X_train_ohe = X_train_ohe.fillna(train_median)
X_val_ohe   = X_val_ohe.fillna(train_median)
X_test_ohe  = X_test_ohe.fillna(train_median)

#  Feature Scaling 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_ohe)
X_val_scaled   = scaler.transform(X_val_ohe)
X_test_scaled  = scaler.transform(X_test_ohe)

#  Class Weights 
cw_arr = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train.values)
class_weight_dict = {0: cw_arr[0], 1: cw_arr[1]}

print(f'One-hot features: {X_train_scaled.shape[1]}')
print(f'Class weights: 0={class_weight_dict[0]:.4f}, 1={class_weight_dict[1]:.4f}')


### 12a. MLP Architecture & Training

In [ ]:
def build_mlp(input_dim, lr=0.001):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.4),
        layers.Dense(64, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.3),
        layers.Dense(32, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auroc', curve='ROC'),
                 keras.metrics.AUC(name='auprc', curve='PR')]
    )
    return model

tf_model = build_mlp(X_train_scaled.shape[1])
tf_model.summary()

#  Training 
early_stop = callbacks.EarlyStopping(monitor='val_auprc', mode='max', patience=15, restore_best_weights=True, verbose=1)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_auprc', mode='max', factor=0.5, patience=7, min_lr=1e-6, verbose=1)

print('\nTraining TensorFlow MLP. ')
history = tf_model.fit(
    X_train_scaled, y_train.values,
    validation_data=(X_val_scaled, y_val.values),
    epochs=150, batch_size=512,
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr], verbose=1
)
print(f'\n  Training completed in {len(history.history["loss"])} epochs')
print(f'   Best val_auprc: {max(history.history["val_auprc"]):.4f}')


### 12b. Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, title in zip(axes, ['loss','auroc','auprc'], ['Loss','AUROC','AUPRC']):
    ax.plot(history.history[metric], label='Train', color='#2563eb', lw=2)
    ax.plot(history.history[f'val_{metric}'], label='Validation', color='#dc2626', lw=2)
    ax.set_xlabel('Epoch'); ax.set_ylabel(title); ax.set_title(title)
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('TensorFlow MLP — Training History', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('tf_mlp_training_history_xgb.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Saved: tf_mlp_training_history_xgb.png')


### 12c. TensorFlow MLP — Evaluation & Bootstrap CIs

In [ ]:
y_train_proba_tf = tf_model.predict(X_train_scaled, verbose=0).flatten()
y_val_proba_tf   = tf_model.predict(X_val_scaled, verbose=0).flatten()
y_test_proba_tf  = tf_model.predict(X_test_scaled, verbose=0).flatten()

tf_results = {}
print('=' * 65)
print(f'  TensorFlow MLP  Results on All Sets')
print('=' * 65)
print(f'  {"Set":<12} {"AUROC":>8} {"AUPRC":>8} {"Brier":>8}')
print('-' * 65)
for name, (yt, yp) in [('Train',(y_train,y_train_proba_tf)),('Validation',(y_val,y_val_proba_tf)),('Test',(y_test,y_test_proba_tf))]:
    a,p,b = roc_auc_score(yt,yp), average_precision_score(yt,yp), brier_score_loss(yt,yp)
    tf_results[name] = {'AUROC':a,'AUPRC':p,'Brier':b}
    print(f'  {name:<12} {a:>8.4f} {p:>8.4f} {b:>8.4f}')
print('=' * 65)

tf_gap = tf_results['Train']['AUROC'] - tf_results['Test']['AUROC']
print(f'\n  Overfitting Gap AUROC: {tf_gap:+.4f}', '✓ Minimal' if tf_gap < 0.05 else ('⚠ Moderate' if tf_gap < 0.10 else '✗ Significant'))

# Bootstrap CIs
print('\nComputing 95% CI for TensorFlow MLP. ')
tf_boot_a, tf_boot_p, tf_boot_b = [], [], []
for i in range(n_bootstrap):
    idx = resample(range(len(y_test)), random_state=i)
    yb, pb = y_test.iloc[idx], y_test_proba_tf[idx]
    if len(yb.unique()) < 2: continue
    tf_boot_a.append(roc_auc_score(yb,pb))
    tf_boot_p.append(average_precision_score(yb,pb))
    tf_boot_b.append(brier_score_loss(yb,pb))

tf_auroc_ci = np.percentile(tf_boot_a,[2.5,97.5])
tf_auprc_ci = np.percentile(tf_boot_p,[2.5,97.5])
tf_brier_ci = np.percentile(tf_boot_b,[2.5,97.5])
tf_test_auroc = tf_results['Test']['AUROC']
tf_test_auprc = tf_results['Test']['AUPRC']
tf_test_brier = tf_results['Test']['Brier']

print('\n' + '=' * 65)
print('  TF MLP  TEST RESULTS WITH 95% CONFIDENCE INTERVALS')
print('=' * 65)
print(f'  AUROC:  {tf_test_auroc:.4f}  (95% CI: {tf_auroc_ci[0]:.4f} – {tf_auroc_ci[1]:.4f})')
print(f'  AUPRC:  {tf_test_auprc:.4f}  (95% CI: {tf_auprc_ci[0]:.4f} – {tf_auprc_ci[1]:.4f})')
print(f'  Brier:  {tf_test_brier:.4f}  (95% CI: {tf_brier_ci[0]:.4f} – {tf_brier_ci[1]:.4f})')
print('=' * 65)


### 12d. ROC & PR Curves — XGBoost vs TensorFlow MLP

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fpr_xg, tpr_xg, _ = roc_curve(y_test, y_test_proba)
fpr_tf, tpr_tf, _ = roc_curve(y_test, y_test_proba_tf)
axes[0].plot(fpr_xg, tpr_xg, color='#2563eb', lw=2, label=f'XGBoost (AUROC = {test_auroc:.4f})')
axes[0].plot(fpr_tf, tpr_tf, color='#f97316', lw=2, linestyle='--', label=f'TF MLP (AUROC = {tf_test_auroc:.4f})')
axes[0].plot([0,1],[0,1],'k--',lw=1,alpha=0.3,label='Random')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC Curve — XGBoost vs TF MLP')
axes[0].legend(loc='lower right'); axes[0].grid(True, alpha=0.3)

p_xg, r_xg, _ = precision_recall_curve(y_test, y_test_proba)
p_tf, r_tf, _ = precision_recall_curve(y_test, y_test_proba_tf)
axes[1].plot(r_xg, p_xg, color='#2563eb', lw=2, label=f'XGBoost (AUPRC = {test_auprc:.4f})')
axes[1].plot(r_tf, p_tf, color='#f97316', lw=2, linestyle='--', label=f'TF MLP (AUPRC = {tf_test_auprc:.4f})')
axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', lw=1, alpha=0.3, label=f'Baseline ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('PR Curve — XGBoost vs TF MLP')
axes[1].legend(loc='upper right'); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('xgb_vs_tf_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Saved: xgb_vs_tf_roc_pr.png')


### 12e. Direct Comparison — XGBoost vs TensorFlow MLP

In [ ]:
print('\n')
print('╔' + '═' * 72 + '╗')
print('║' + '  XGBOOST vs TENSORFLOW MLP  HEAD TO HEAD'.ljust(72) + '║')
print('╠' + '═' * 72 + '╣')
print('║' + ''.ljust(72) + '║')
print('║' + f'  {"Metric":<12} {"XGBoost":>20} {"TF MLP":>20} {"Delta (TF-XGB)":>14}'.ljust(72) + '║')
print('║' + f'  {"-"*12:<12} {"-"*20:>20} {"-"*20:>20} {"-"*14:>14}'.ljust(72) + '║')

for metric, xv, xci, tv, tci in [
    ('AUROC', test_auroc, auroc_ci, tf_test_auroc, tf_auroc_ci),
    ('AUPRC', test_auprc, auprc_ci, tf_test_auprc, tf_auprc_ci),
    ('Brier', test_brier, brier_ci, tf_test_brier, tf_brier_ci)]:
    xs = f'{xv:.4f} ({xci[0]:.4f}-{xci[1]:.4f})'
    ts = f'{tv:.4f} ({tci[0]:.4f}-{tci[1]:.4f})'
    d = tv - xv
    print('║' + f'  {metric:<12} {xs:>20} {ts:>20} {d:>+14.4f}'.ljust(72) + '║')

print('║' + ''.ljust(72) + '║')

ov_auroc = (auroc_ci[0] <= tf_auroc_ci[1]) and (tf_auroc_ci[0] <= auroc_ci[1])
ov_auprc = (auprc_ci[0] <= tf_auprc_ci[1]) and (tf_auprc_ci[0] <= auprc_ci[1])

for name, ov, xv, tv in [('AUROC', ov_auroc, test_auroc, tf_test_auroc), ('AUPRC', ov_auprc, test_auprc, tf_test_auprc)]:
    if not ov:
        w = 'TF MLP' if tv > xv else 'XGBoost'
        print('║' + f'   {name} 95% CIs do NOT overlap -> {w} statistically better.'.ljust(72) + '║')
    else:
        print('║' + f'    {name} 95% CIs overlap -> no significant difference.'.ljust(72) + '║')

print('║' + ''.ljust(72) + '║')
print('║' + '  Both models converge near the ~0.667 AUROC ceiling  a dataset'.ljust(72) + '║')
print('║' + '  property due to noisy clinical records (Strack et al., 2014).'.ljust(72) + '║')
print('║' + '  Feature engineering remains the primary performance driver.'.ljust(72) + '║')
print('║' + ''.ljust(72) + '║')
print('╚' + '═' * 72 + '╝')


### 12f. TF MLP — Threshold Analysis & Confusion Matrix

In [ ]:
pt, rt, tt = precision_recall_curve(y_test, y_test_proba_tf)
f1t = 2*(pt*rt)/(pt+rt+1e-8)
ti = np.argmax(f1t); tf_thresh = tt[ti]
print(f'TF MLP  Optimal threshold: {tf_thresh:.4f}, F1: {f1t[ti]:.4f}\n')
yp_tf = (y_test_proba_tf >= tf_thresh).astype(int)
print(classification_report(y_test, yp_tf, target_names=['No Readmission','Readmission <30d']))

cm_tf = confusion_matrix(y_test, yp_tf)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No','Yes'], yticklabels=['No','Yes'], ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual'); axes[0].set_title(f'XGBoost (t={optimal_threshold:.4f})')
sns.heatmap(cm_tf, annot=True, fmt='d', cmap='Oranges', xticklabels=['No','Yes'], yticklabels=['No','Yes'], ax=axes[1])
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual'); axes[1].set_title(f'TF MLP (t={tf_thresh:.4f})')
plt.suptitle('Confusion Matrices — XGBoost vs TF MLP', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig('xgb_vs_tf_confusion.png', dpi=150, bbox_inches='tight'); plt.show()
print(' Saved: xgb_vs_tf_confusion.png')


### 12g. Save TensorFlow Model

In [ ]:
tf_model.save('tf_mlp_readmission_xgb.keras')
print(' TensorFlow model saved: tf_mlp_readmission_xgb.keras')
tf_model.export('tf_mlp_savedmodel_xgb')
print(' SavedModel saved: tf_mlp_savedmodel_xgb/')
joblib.dump(y_test_proba_tf, 'tf_mlp_test_predictions_xgb.pkl')
print(' Predictions saved: tf_mlp_test_predictions_xgb.pkl')


## 13. Clinical Decision Support — Dual Reports

In [ ]:
RISK_THRESHOLD = 0.30  # Clinical threshold

def generate_clinical_report(patient_idx, probability):
    risk = 'HIGH' if probability >= RISK_THRESHOLD else 'LOW'
    report = f"""
╔══════════════════════════════════════════════════════════════╗
║          CLINICAL DECISION SUPPORT REPORT                  ║
╚══════════════════════════════════════════════════════════════╝

Patient Encounter: #{patient_idx}
Predicted Readmission Probability: {probability:.4f} ({probability*100:.1f}%)
Risk Classification: {risk} RISK
Decision Threshold: {RISK_THRESHOLD}
"""
    if risk == 'HIGH':
        report += """
──── HIGH-RISK STANDARDIZED CHECKLIST ────
□ Review discharge medication reconciliation
□ Schedule follow-up appointment within 7 days
□ Assess patient understanding of discharge instructions
□ Evaluate social support and transportation barriers
□ Review HbA1c and adjust diabetes management plan
□ Consider referral to care coordination / case management
"""
    report += f"""
DISCLAIMER: Decision-support tool only. Clinical judgment
should always take precedence over algorithmic predictions.
Model: Tuned XGBoost | AUROC: {test_auroc:.4f} | AUPRC: {test_auprc:.4f}
"""
    return report

def generate_patient_report(patient_idx, probability):
    risk = 'higher' if probability >= RISK_THRESHOLD else 'lower'
    report = f"""
╔══════════════════════════════════════════════════════════════╗
║               YOUR AFTER-HOSPITAL CARE SUMMARY             ║
╚══════════════════════════════════════════════════════════════╝

Your estimated risk of returning to the hospital within 30 days: {risk.upper()}
"""
    if risk == 'higher':
        report += """
WHAT YOU CAN DO:
 Take all medications as prescribed
 Attend your follow-up appointment
 Monitor your blood sugar regularly
 Call your doctor if you feel unwell
 Keep a list of all your medications
"""
    else:
        report += """
CONTINUE YOUR GOOD CARE:
 Follow your discharge instructions
 Take medications as prescribed
 Attend scheduled follow-ups
"""
    return report

# Demo: 3 random patients
print("=" * 65)
print("  DEMO: Clinical Decision Support Reports")
print("=" * 65)

np.random.seed(42)
demo_indices = np.random.choice(len(y_test), 3, replace=False)

for idx in demo_indices:
    prob = y_test_proba[idx]
    actual = y_test.iloc[idx]
    print(generate_clinical_report(idx, prob))
    print(f"  [ACTUAL OUTCOME: {'Readmitted' if actual == 1 else 'Not readmitted'}]\n")

## 14. Final Summary

In [ ]:
print('\n')
print('╔' + '═' * 72 + '╗')
print('║' + '  FINAL RESULTS  XGBoost + TENSORFLOW MLP COMPARISON'.ljust(72) + '║')
print('╠' + '═' * 72 + '╣')
print('║' + ''.ljust(72) + '║')
print('║' + '   XGBoost (Primary Model) '.ljust(72) + '║')
print('║' + f'  AUROC:  {test_auroc:.4f}  (95% CI: {auroc_ci[0]:.4f} – {auroc_ci[1]:.4f})'.ljust(72) + '║')
print('║' + f'  AUPRC:  {test_auprc:.4f}  (95% CI: {auprc_ci[0]:.4f} – {auprc_ci[1]:.4f})'.ljust(72) + '║')
print('║' + f'  Brier:  {test_brier:.4f}  (95% CI: {brier_ci[0]:.4f} – {brier_ci[1]:.4f})'.ljust(72) + '║')
print('║' + ''.ljust(72) + '║')
print('║' + '   TensorFlow MLP '.ljust(72) + '║')
print('║' + f'  AUROC:  {tf_test_auroc:.4f}  (95% CI: {tf_auroc_ci[0]:.4f} – {tf_auroc_ci[1]:.4f})'.ljust(72) + '║')
print('║' + f'  AUPRC:  {tf_test_auprc:.4f}  (95% CI: {tf_auprc_ci[0]:.4f} – {tf_auprc_ci[1]:.4f})'.ljust(72) + '║')
print('║' + f'  Brier:  {tf_test_brier:.4f}  (95% CI: {tf_brier_ci[0]:.4f} – {tf_brier_ci[1]:.4f})'.ljust(72) + '║')
print('║' + ''.ljust(72) + '║')
print('║' + f'  AUROC Delta (TF - XGB): {tf_test_auroc - test_auroc:+.4f}'.ljust(72) + '║')
print('║' + f'  AUPRC Delta (TF - XGB): {tf_test_auprc - test_auprc:+.4f}'.ljust(72) + '║')
print('║' + ''.ljust(72) + '║')
print('║' + '  Published Benchmark (same dataset): AUROC = 0.667'.ljust(72) + '║')
print('║' + ''.ljust(72) + '║')
print('╠' + '═' * 72 + '╣')
print('║' + '  Methodological Features:'.ljust(72) + '║')
print('║' + '     Patient-level split (GroupShuffleSplit)'.ljust(72) + '║')
print('║' + '     Optuna tuning (50 trials + regularization params)'.ljust(72) + '║')
print('║' + '     TensorFlow MLP (128->64->32, BatchNorm, Dropout)'.ljust(72) + '║')
print('║' + '     Cross-validated calibration (isotonic, 3-fold)'.ljust(72) + '║')
print('║' + '     Bootstrap 95% confidence intervals (both models)'.ljust(72) + '║')
print('║' + '     SHAP interpretability analysis'.ljust(72) + '║')
print('║' + '     Interaction features + missing data flags'.ljust(72) + '║')
print('║' + ''.ljust(72) + '║')
print('║' + '  Conclusion:'.ljust(72) + '║')
print('║' + '    Both models converge near the ~0.667 AUROC ceiling.'.ljust(72) + '║')
print('║' + '    Well-tuned tree-based models match or outperform'.ljust(72) + '║')
print('║' + '    neural networks on tabular clinical data'.ljust(72) + '║')
print('║' + '    (Grinsztajn et al., 2022; Shwartz-Ziv & Armon, 2022).'.ljust(72) + '║')
print('║' + ''.ljust(72) + '║')
print('╚' + '═' * 72 + '╝')
